# Time aggregation: the manager sentiment series

Takes the report-level polarities from `sentiment_extraction_results/`, collapses
them to one observation per calendar month, smooths them, and writes the series
to `manager_sentiment_results/`.

Both steps are configurable:

| step | options |
|---|---|
| weighting | equal, or by any report-level column (market capitalization) |
| pooling | standardize within `FORM_TYPE` before pooling, or pool raw |
| smoothing | `none`, `ma` (backward window), `hp` (two-sided or one-sided) |

Two choices here matter for what the analysis stage can do with the output, so
they are spelled out below rather than buried in the code: **standardizing
within form type**, and **which smoothed series may be used as a predictor**.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]  # code/time aggregation -> code -> repo root
sys.path.insert(0, str(REPO_ROOT / "code" / "scripts"))

import matplotlib.pyplot as plt
import pandas as pd

import time_aggregation as ta
from utils import MANAGER_SENTIMENT_DIR, SENTIMENT_RESULTS_DIR, set_seed

pd.set_option("display.max_columns", None)
set_seed(42)

print("HP lambda for monthly data:", ta.HP_LAMBDA_MONTHLY)

## Report-level input

Every file the extraction stage wrote carries its own `strategy` and `model`
columns, so several runs can be concatenated and then pivoted into one column
per measure.

In [ ]:
files = sorted(
    p for p in SENTIMENT_RESULTS_DIR.glob("*.csv")
    if not p.stem.startswith("_") and ".failures" not in p.name
)
print("\n".join(f"  {p.name}" for p in files) or "  (no extraction output yet)")

reports = pd.concat([pd.read_csv(p, dtype={"CIK": str, "FILING_DATE": str, "ACC_NUM": str}) for p in files],
                    ignore_index=True)
print()
print(reports.groupby(["strategy", "FORM_TYPE"]).size().to_string())

In [ ]:
# One column per measure, named polarity_<strategy>, keyed by the filing.
KEY = ["CIK", "FILING_DATE", "ACC_NUM", "FORM_TYPE"]

wide = (
    reports.pivot_table(index=KEY, columns="strategy", values="polarity", aggfunc="mean")
    .rename(columns=lambda s: f"polarity_{s}")
    .reset_index()
)

MEASURES = [c for c in wide.columns if c.startswith("polarity_")]
print("measures:", MEASURES)
print("filings :", len(wide))
wide[MEASURES].describe().round(4)

## Why equalize across form types

10-K and 10-Q filings do not arrive evenly across the calendar. If the two form
types sit at different levels or spreads, a plain pooled monthly mean picks up
the calendar composition as a mechanical annual cycle, and a regression later
reads that cycle as signal.

Two ways to remove it, and the choice is substantive:

- `standardize_method="zscore"` equalizes level **and** spread. It stops months
  dominated by the more dispersed form type from being mechanically more
  volatile, but it rescales the measure: a form type whose scores are tightly
  clustered gets stretched more than a dispersed one.
- `standardize_method="center"` removes only the level difference and leaves each
  form type's spread untouched.

The cells below show the composition, the two distributions and what each choice
does, so the decision rests on this sample rather than on a default.

In [ ]:
wide["month"] = wide["FILING_DATE"].str[4:6]

composition = pd.crosstab(wide["month"], wide["FORM_TYPE"])
print("filings by calendar month and form type")
print(composition.to_string())
print()
print("score distribution by form type")
print(wide.groupby("FORM_TYPE")[MEASURES].agg(["mean", "std", "count"]).round(4).to_string())

In [ ]:
STANDARDIZE_BY = "FORM_TYPE"      # None to pool the raw scores instead
STANDARDIZE_METHOD = "zscore"     # "zscore" = level + scale, "center" = level only
WEIGHT_COL = None                 # e.g. "mktcap" once market capitalization is merged in

pooled = ta.aggregate_to_monthly(wide, MEASURES, standardize_by=None)
within = ta.aggregate_to_monthly(wide, MEASURES, standardize_by="FORM_TYPE",
                                 standardize_method=STANDARDIZE_METHOD)

measure = MEASURES[0]
by_calendar_month = pd.DataFrame(
    {
        "pooled": pooled.set_index(ta.MONTH_COL)[measure].groupby(lambda d: d.month).mean(),
        "equalized": within.set_index(ta.MONTH_COL)[measure].groupby(lambda d: d.month).mean(),
    }
)
print(f"average {measure} by calendar month")
print(by_calendar_month.round(4).to_string())
print()
print("month-to-month volatility of the monthly index (std of first differences):")
for label, frame in [("pooled", pooled), ("equalized", within)]:
    print(f"  {label:10s} {frame[measure].diff().std():.4f}")

## Market-cap weighting

`aggregate_to_monthly` weights by any report-level column. Market capitalization
is not part of the extraction output, so merge it onto the filing key first;
without it the aggregation is equal-weighted and says so in the log.

The weight is taken as of the filing, which is information available at month
*t* -- fine for a predictor built at *t*.

In [ ]:
# MKTCAP_PATH = REPO_ROOT / "input_data" / "market_cap.csv"   # CIK, FILING_DATE, mktcap
#
# mktcap = pd.read_csv(MKTCAP_PATH, dtype={"CIK": str, "FILING_DATE": str})
# wide = wide.merge(mktcap, on=["CIK", "FILING_DATE"], how="left")
# print("weight coverage:", wide["mktcap"].notna().mean().round(3))
# WEIGHT_COL = "mktcap"

print("weighting:", WEIGHT_COL or "equal")

## Build the series

One column per measure and smoother: no suffix for the raw monthly mean, `_ma4`
for the 4-month backward average, `_hp` for the two-sided HP trend, `_hp1s` for
the one-sided one.

**The `_hp` columns are look-ahead.** The two-sided filter builds the trend at
month *t* out of the whole sample, including months after *t*. That is what makes
it a good description of the series, and what disqualifies it as a predictor of
later returns. For the predictive regressions use the raw series, `_ma4`, or
`_hp1s`, which re-estimates the filter on an expanding window and keeps only its
last point.

In [ ]:
SMOOTHING = [
    ("none", {}),
    ("ma", {"window": 4}),
    ("hp", {}),                                  # two-sided: descriptive only
    ("hp", {"one_sided": True, "min_periods": 24}),
]

series = ta.build_manager_sentiment(
    wide,
    value_cols=MEASURES,
    weight_col=WEIGHT_COL,
    standardize_by=STANDARDIZE_BY,
    standardize_method=STANDARDIZE_METHOD,
    smoothing=SMOOTHING,
)

print(series.shape)
series.tail()

In [ ]:
fig, axes = plt.subplots(len(MEASURES), 1, figsize=(13, 4 * len(MEASURES)), squeeze=False)

for ax, measure in zip(axes[:, 0], MEASURES):
    ax.plot(series[ta.MONTH_COL], series[measure], alpha=0.35, linewidth=1, label="monthly")
    ax.plot(series[ta.MONTH_COL], series[f"{measure}_ma4"], linewidth=2, label="4-month MA")
    ax.plot(series[ta.MONTH_COL], series[f"{measure}_hp"], linewidth=2, label="HP trend (two-sided)")
    ax.plot(series[ta.MONTH_COL], series[f"{measure}_hp1s"], linewidth=2, linestyle="--",
            label="HP trend (one-sided)")
    ax.set_title(measure)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# How many filings each monthly point rests on -- thin months are noisy.
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(series[ta.MONTH_COL], series["n_obs"], width=20)
ax.set_title("filings per month")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(series["n_obs"].describe().round(1).to_string())

## Write the output

In [ ]:
OUTPUT_PATH = MANAGER_SENTIMENT_DIR / "manager_sentiment_monthly.csv"
ta.save_time_series(series, OUTPUT_PATH)

print()
print("columns:")
for col in series.columns:
    print("  -", col)